# Disaster XAI-Net — EfficientNet, 5-Fold CV + Held-Out Test

Two-stage satellite damage assessment, EfficientNet-B0 backbone, siamese over pre/post pairs.

```
pre-disaster tile + post-disaster tile
                 |
   Stage 1: hazard classification
   cyclone / earthquake / flood / wildfire
                 |
   Stage 2: damage severity (per building)
   no-damage / minor / major / destroyed
                 |
        rescue prioritization
```

### What changed from the previous version

| | before | now |
|---|---|---|
| CV pool | `train/` + `test/` pooled | **`train/` only** |
| held-out test | none | **`test/` untouched until the end** |
| max epochs | 10 | **40, early stopping** |
| metrics | macro-F1 | accuracy, P/R/F1, ROC-AUC, MCC, kappa, QWK |
| severity views | multiclass | **multiclass + binary** |
| per-fold outputs | summary only | curves, ROC, confusion, class-wise, CSVs |
| explainability | none | **Grad-CAM** |
| final model | none | trained on full dev, evaluated once on test |

### Read this before running

Restricting cross-validation to `train/` changes the fold pool from 2668 pairs to 2131,
which changes both fold checksums. **Results from the earlier pooled-set runs are not
comparable to these** — every architecture has to be re-run against the new checksums
printed below. There is no way to mix the two.

---
## 1. Experimental design — why the data is split this way

The preprocessing notebook already produced an ~80:20 split at the **pre/post image-pair
level**, stored as `hazard_data/<hazard>/{train,test}/`. That split is taken as given and
is never regenerated here.

```
                    EXISTING DATA (2668 pairs)
                              |
                ┌─────────────┴─────────────┐
                |                           |
        train/  (~80%, 2131)        test/  (~20%, 537)
                |                           |
                ▼                           |
       5-fold cross-validation              |  untouched
                |                           |  until the
    ┌───────┬───┴───┬───────┐               |  very end
  fold 1  fold 2  ...    fold 5             |
                |                           |
                └─────────────┬─────────────┘
                              ▼
                    FINAL HELD-OUT TEST
```

**What 80/20 then 5-fold means in proportions.** The 80% development portion is divided
into five folds; each iteration trains on four and validates on one. Relative to the
complete dataset that is roughly:

* **64%** training
* **16%** validation
* **20%** test, untouched

**The 20% test set is never used for** training, validation, hyperparameter tuning, early
stopping, model selection, augmentation decisions, or threshold selection. It is read
exactly once, in the final section, after every cross-validation decision has been frozen.

**Why hold it out at all?** Cross-validation estimates how well the *procedure*
generalizes, but the procedure itself consumes the validation folds — early stopping and
best-epoch selection both read validation scores. A held-out set that influenced nothing
is the only clean estimate of what the finished model does on unseen data.

**Why pre/post pairs stay together.** One row of the manifest *is* a pair: both images
share a fold by construction, so a tile can never have its pre image in training and its
post image in validation. For Stage 2 the folds are additionally grouped on the source
tile, because crops from one tile are near-duplicates of each other. Both properties are
asserted, not assumed.

---
## 2. Imports and configuration

In [ ]:
import os, re, gc, json, time, random, hashlib, itertools, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, auc, matthews_corrcoef,
                             cohen_kappa_score, confusion_matrix, classification_report)
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("torch", torch.__version__, "| timm", timm.__version__, "| pandas", pd.__version__)

### Configuration

`MAX_EPOCHS = 40` with early stopping, as specified. Training stops early when validation
macro-F1 has not improved for `PATIENCE` epochs; it never runs past 40. Every fold records
its best epoch, how many epochs actually ran, and whether early stopping fired.

Selection is on validation **macro-F1** rather than accuracy or loss, because both stages
are imbalanced — no-damage dominates Stage 2 and cyclone is ~46% of Stage 1, so accuracy
rewards a model that neglects the rare classes.

In [ ]:
# ---------------- identity ----------------
ARCH_TAG    = "effnet"
S1_MODEL    = "efficientnet_b0"
S2_MODEL    = "efficientnet_b0"
LR_BACKBONE = 1e-4          # standard CNN fine-tuning rate (a ViT would collapse here)
LR_HEAD     = 1e-3

# ---------------- protocol (must match across the team) ----------------
N_SPLITS   = 5
SEED       = 42
MAX_EPOCHS = 40             # professor's requirement
PATIENCE   = 8              # early stopping patience on validation macro-F1

S1_IMG,  S1_BATCH  = 224, 32
S2_CROP, S2_BATCH  = 64, 256

MAX_PER_CLASS     = 10000   # cap on the majority damage class (dev set only)
BBOX_PAD          = 0.25
DROP_UNCLASSIFIED = True

USE_PAIR     = True         # siamese: shared backbone over pre + post
DROPOUT      = 0.3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.05
NUM_WORKERS  = 2

# ---------------- output ----------------
RESULTS_DIR = "/kaggle/working/effnet_results"
SUBDIRS = ["stage1_hazard", "stage2_severity_multiclass", "stage2_severity_binary",
           "gradcam", "final_test"]
for s in SUBDIRS:
    for f in range(1, N_SPLITS + 1):
        os.makedirs(os.path.join(RESULTS_DIR, s, f"fold_{f}"), exist_ok=True)
    os.makedirs(os.path.join(RESULTS_DIR, s), exist_ok=True)

HAZARD_NAMES = ['cyclone', 'earthquake', 'flood', 'wildfire']
DAMAGE_NAMES = ['no-damage', 'minor-damage', 'major-damage', 'destroyed']
BINARY_NAMES = ['no-damage', 'damaged']
hazard_to_idx = {n: i for i, n in enumerate(HAZARD_NAMES)}
damage_to_idx = {n: i for i, n in enumerate(DAMAGE_NAMES)}

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(SEED)
AMP = torch.cuda.is_available()
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD  = [0.229, 0.224, 0.225]

def checksum(keys, folds):
    s = "|".join(f"{k}:{f}" for k, f in sorted(zip(map(str, keys), map(int, folds))))
    return hashlib.md5(s.encode()).hexdigest()[:16]

print(f"{ARCH_TAG} | MAX_EPOCHS={MAX_EPOCHS} patience={PATIENCE} | results -> {RESULTS_DIR}")

---
## 3. Load the existing split (never regenerated)

In [ ]:
hazard_base_path = None
for b in ["/kaggle/input", "/content"]:
    if os.path.isdir(b):
        for dp, _, _ in os.walk(b):
            if os.path.basename(dp) == "hazard_data":
                hazard_base_path = dp; break
    if hazard_base_path: break
assert hazard_base_path, ("hazard_data not found. On Kaggle: + Add Input > "
                          "Notebook Output > lubdham/disaster-separate-data")
print("hazard_base_path:", hazard_base_path)


def get_full_prefix(fn):
    m = re.match(r'^(.*?)_(pre|post)_disaster\.png$', fn)
    return m.group(1) if m else None

def get_event_name(prefix):
    m = re.match(r'^(.*)_\d+$', prefix)
    return m.group(1) if m else prefix

records = []
for hazard in sorted(os.listdir(hazard_base_path)):
    for split in ['train', 'test']:                 # the EXISTING split, preserved
        img_dir = os.path.join(hazard_base_path, hazard, split, 'images')
        lbl_dir = os.path.join(hazard_base_path, hazard, split, 'labels')
        if not os.path.isdir(img_dir):
            continue
        files = set(os.listdir(img_dir))
        for p in sorted({get_full_prefix(f) for f in files if get_full_prefix(f)}):
            pre_f, post_f = f"{p}_pre_disaster.png", f"{p}_post_disaster.png"
            if pre_f in files and post_f in files:  # pair kept together by construction
                records.append({
                    'hazard_type': hazard, 'orig_split': split, 'event_prefix': p,
                    'pre_path':  os.path.join(img_dir, pre_f),
                    'post_path': os.path.join(img_dir, post_f),
                    'post_label_path': os.path.join(lbl_dir, post_f.replace('.png', '.json')),
                })

all_df = (pd.DataFrame(records).sort_values('event_prefix').reset_index(drop=True)
            .sample(frac=1.0, random_state=SEED).reset_index(drop=True))
all_df['event_name'] = all_df['event_prefix'].apply(get_event_name)
all_df['label'] = all_df['hazard_type'].map(hazard_to_idx)

dev_df  = all_df[all_df.orig_split == 'train'].reset_index(drop=True)   # 5-fold CV pool
test_df = all_df[all_df.orig_split == 'test'].reset_index(drop=True)    # untouched

print(f"\ntotal pairs {len(all_df)} | development {len(dev_df)} "
      f"({len(dev_df)/len(all_df):.1%}) | held-out test {len(test_df)} "
      f"({len(test_df)/len(all_df):.1%})")
print("\nper hazard:")
print(pd.crosstab(all_df['hazard_type'], all_df['orig_split'], margins=True))
print("\nevents per hazard:")
for h, g in all_df.groupby('hazard_type'):
    print(f"  {h:11s} {g['event_name'].nunique()} event(s): {sorted(g['event_name'].unique())}")

### Leakage checks

Three properties are asserted rather than assumed. If any fails, execution stops.

1. **Pre/post pairs** — one manifest row holds both images of a tile, so they cannot be
   separated. Verified by checking that every prefix appears exactly once and that both
   files exist.
2. **Development vs test disjoint** — no tile prefix appears in both.
3. **Group leakage** (checked later, after folds are built) — no tile appears in both the
   training and validation side of any Stage 2 fold.

In [ ]:
# 1. pre/post pairs inseparable
assert not all_df['event_prefix'].duplicated().any(), "duplicate tile prefixes"
assert all_df['pre_path'].map(os.path.exists).all(),  "missing pre image"
assert all_df['post_path'].map(os.path.exists).all(), "missing post image"
print("[OK] No pre/post leakage - every pair is one row and shares a fold by construction")

# 2. dev and test disjoint
overlap = set(dev_df['event_prefix']) & set(test_df['event_prefix'])
assert not overlap, f"{len(overlap)} tiles appear in BOTH development and test"
assert len(dev_df) + len(test_df) == len(all_df)
print("[OK] Test set disjoint from development set")
print(f"[OK] Test set untouched - {len(test_df)} pairs reserved, read only in section 10")

---
## 4. Preprocessing vs augmentation

```
RAW IMAGE
   |
   v
COMMON PREPROCESSING          (resize, tensor, ImageNet normalisation)
   |
   +---------------+
   |               |
   v               v
TRAINING       VALIDATION / TEST
   |               |
   v               v
AUGMENTATION    NO AUGMENTATION
```

**Training** gets resize to 256, random horizontal and vertical flip, rotation up to 20
degrees, mild colour jitter, then a random 224 crop. Satellite imagery has no canonical
orientation — a building is a building whether the tile is flipped or rotated — so the
flips and rotations are label-preserving. Brightness and contrast jitter is kept mild
because it stands in for sensor and illumination variation, and pushed too far it would
destroy the very cues that separate scorch marks from floodwater.

**Validation and test** get resize to 256, deterministic centre crop to 224, tensor,
normalisation. Nothing random.

**Why validation must not be augmented.** Augmentation exists to enlarge the training
distribution. Applying it at validation time makes the score depend on which random draws
happened to be sampled, so the number moves between runs on identical weights, early
stopping fires on noise, and the reported metric measures performance on distorted images
rather than on real ones. The same reasoning applies with more force to the test set.

For Stage 2 crops the equivalent applies: training uses the 8 dihedral transforms
(4 rotations x optional flip), validation and test use none.

In [ ]:
# Stage 1 (tiles) -------------------------------------------------------
s1_train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomCrop(S1_IMG),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

s1_eval_tf = transforms.Compose([                # validation AND test - deterministic
    transforms.Resize((256, 256)),
    transforms.CenterCrop(S1_IMG),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

_MEAN_T = torch.tensor(NORM_MEAN).view(3, 1, 1)
_STD_T  = torch.tensor(NORM_STD).view(3, 1, 1)

print("train transform :", len(s1_train_tf.transforms), "ops (augmented)")
print("eval  transform :", len(s1_eval_tf.transforms), "ops (deterministic, no augmentation)")

---
## 5. Metrics

**Accuracy, precision, recall, F1** — reported macro (unweighted mean over classes, so a
rare class counts as much as a common one) and weighted (by support).

**ROC-AUC** — one-vs-rest for the 4-class problems, macro and weighted averaged, computed
from predicted probabilities rather than hard labels. Per-class, micro-average and
macro-average ROC curves are plotted for every fold.

**MCC (Matthews correlation coefficient)** — a single correlation-like number between the
true and predicted labels, ranging -1 to +1 with 0 meaning chance. Unlike accuracy it
cannot be inflated by predicting the majority class, which makes it the right summary
statistic for imbalanced problems. The multiclass generalisation is used here.

**Cohen's kappa** — agreement between prediction and ground truth corrected for the
agreement expected by chance. 0 means no better than guessing at the observed class rates.

**QWK (quadratic weighted kappa)** — kappa where disagreements are penalised by the
*square* of the distance between the classes. This is appropriate for Stage 2 and only for
Stage 2, because damage severity is **ordinal**:

```
no-damage < minor-damage < major-damage < destroyed
```

Confusing minor with major is a small error; confusing no-damage with destroyed is a
serious one, and for rescue prioritisation those two mistakes are not equally costly.
Plain accuracy and macro-F1 treat them identically. QWK does not.

QWK is **not** computed for Stage 1, because cyclone / earthquake / flood / wildfire have
no natural ordering — any ordering would be arbitrary, and the quadratic penalty would
encode a distance that does not exist.

**Class-wise accuracy.** For multiclass problems the per-class quantity that behaves like
accuracy is **recall** (of the true members of this class, how many were found). A
per-class one-vs-rest accuracy is also reported, with the caveat that it is dominated by
the true negatives of the other three classes and so looks high for every rare class.

In [ ]:
def compute_metrics(y, pred, prob, n_classes, ordinal=False):
    """All scalar metrics for one (fold, stage, view)."""
    m = {}
    m['accuracy'] = accuracy_score(y, pred)
    for avg in ['macro', 'weighted']:
        m[f'precision_{avg}'] = precision_score(y, pred, average=avg, zero_division=0)
        m[f'recall_{avg}']    = recall_score(y, pred, average=avg, zero_division=0)
        m[f'f1_{avg}']        = f1_score(y, pred, average=avg, zero_division=0)
    try:
        if n_classes == 2:
            m['roc_auc'] = roc_auc_score(y, prob[:, 1])
            m['roc_auc_macro'] = m['roc_auc']
            m['roc_auc_weighted'] = m['roc_auc']
        else:
            m['roc_auc_macro'] = roc_auc_score(y, prob, multi_class='ovr', average='macro')
            m['roc_auc_weighted'] = roc_auc_score(y, prob, multi_class='ovr',
                                                  average='weighted')
            m['roc_auc'] = m['roc_auc_macro']
    except ValueError as e:
        m['roc_auc'] = m['roc_auc_macro'] = m['roc_auc_weighted'] = np.nan
        print("   ROC-AUC unavailable:", e)
    m['mcc'] = matthews_corrcoef(y, pred)
    m['cohen_kappa'] = cohen_kappa_score(y, pred)
    if ordinal:
        m['qwk'] = cohen_kappa_score(y, pred, weights='quadratic')
    return m


def classwise_table(y, pred, names):
    """Per-class precision / recall / F1 / support + one-vs-rest accuracy."""
    rep = classification_report(y, pred, labels=list(range(len(names))),
                                target_names=names, output_dict=True, zero_division=0)
    cm = confusion_matrix(y, pred, labels=list(range(len(names))))
    total = cm.sum()
    rows = []
    for i, n in enumerate(names):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = total - tp - fn - fp
        rows.append({'class': n,
                     'precision': rep[n]['precision'], 'recall': rep[n]['recall'],
                     'f1': rep[n]['f1-score'], 'support': int(rep[n]['support']),
                     'ovr_accuracy': (tp + tn) / total})
    for key, label in [('macro avg', 'Macro Avg'), ('weighted avg', 'Weighted Avg')]:
        rows.append({'class': label, 'precision': rep[key]['precision'],
                     'recall': rep[key]['recall'], 'f1': rep[key]['f1-score'],
                     'support': int(rep[key]['support']), 'ovr_accuracy': np.nan})
    return pd.DataFrame(rows)


def plot_training_curves(hist, title, path):
    ep = range(1, len(hist['train_loss']) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(ep, hist['train_acc'], marker='o', ms=3, label='training accuracy')
    ax[0].plot(ep, hist['val_acc'],   marker='s', ms=3, label='validation accuracy')
    ax[0].set_xlabel('epoch'); ax[0].set_ylabel('accuracy')
    ax[0].set_title(f'{title} - accuracy'); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(ep, hist['train_loss'], marker='o', ms=3, label='training loss')
    ax[1].plot(ep, hist['val_loss'],   marker='s', ms=3, label='validation loss')
    ax[1].set_xlabel('epoch'); ax[1].set_ylabel('loss')
    ax[1].set_title(f'{title} - loss'); ax[1].legend(); ax[1].grid(alpha=.3)
    if 'best_epoch' in hist:
        for a in ax:
            a.axvline(hist['best_epoch'], ls='--', c='k', lw=1, alpha=.6)
    plt.tight_layout(); plt.savefig(path, dpi=150, bbox_inches='tight'); plt.show()


def plot_confusion(y, pred, names, title, path):
    cm = confusion_matrix(y, pred, labels=list(range(len(names))))
    cmn = cm.astype(float) / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(1, 2, figsize=(6 + 2.2 * len(names), 4.4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax[0],
                xticklabels=names, yticklabels=names, cbar=False)
    ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('Actual'); ax[0].set_title(f'{title} - counts')
    sns.heatmap(cmn, annot=True, fmt='.3f', cmap='Blues', ax=ax[1], vmin=0, vmax=1,
                xticklabels=names, yticklabels=names, cbar=False)
    ax[1].set_xlabel('Predicted'); ax[1].set_ylabel('Actual')
    ax[1].set_title(f'{title} - row-normalised (recall)')
    plt.tight_layout(); plt.savefig(path, dpi=150, bbox_inches='tight'); plt.show()
    return cm


def plot_roc(y, prob, names, title, path):
    """One-vs-rest ROC with micro and macro averages. Returns dict of AUCs."""
    n = len(names)
    aucs = {}
    plt.figure(figsize=(6.8, 5.4))
    if n == 2:
        fpr, tpr, _ = roc_curve(y, prob[:, 1])
        aucs['damaged'] = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=2, label=f"damaged (AUC {aucs['damaged']:.4f})")
    else:
        Y = label_binarize(y, classes=list(range(n)))
        fprs, tprs = {}, {}
        for i, nm in enumerate(names):
            if Y[:, i].sum() == 0:
                continue
            fprs[i], tprs[i], _ = roc_curve(Y[:, i], prob[:, i])
            aucs[nm] = auc(fprs[i], tprs[i])
            plt.plot(fprs[i], tprs[i], lw=1.6, label=f"{nm} (AUC {aucs[nm]:.4f})")
        fpr_mi, tpr_mi, _ = roc_curve(Y.ravel(), prob.ravel())
        aucs['micro'] = auc(fpr_mi, tpr_mi)
        plt.plot(fpr_mi, tpr_mi, ls=':', lw=2.4, color='deeppink',
                 label=f"micro-average (AUC {aucs['micro']:.4f})")
        grid = np.unique(np.concatenate([fprs[i] for i in fprs]))
        mean_tpr = np.zeros_like(grid)
        for i in fprs:
            mean_tpr += np.interp(grid, fprs[i], tprs[i])
        mean_tpr /= len(fprs)
        aucs['macro'] = auc(grid, mean_tpr)
        plt.plot(grid, mean_tpr, ls=':', lw=2.4, color='navy',
                 label=f"macro-average (AUC {aucs['macro']:.4f})")
    plt.plot([0, 1], [0, 1], 'k--', lw=1, alpha=.5)
    plt.xlim(0, 1); plt.ylim(0, 1.02)
    plt.xlabel('False positive rate'); plt.ylabel('True positive rate')
    plt.title(title); plt.legend(loc='lower right', fontsize=8); plt.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(path, dpi=150, bbox_inches='tight'); plt.show()
    return aucs


def summary_table(fold_metrics, keys):
    """metric x fold1..fold5 + mean + std, as the professor asked."""
    rows = []
    for k in keys:
        vals = [fm.get(k, np.nan) for fm in fold_metrics]
        row = {'Metric': k}
        row.update({f'Fold {i+1}': v for i, v in enumerate(vals)})
        row['Mean'] = np.nanmean(vals); row['Std'] = np.nanstd(vals, ddof=1)
        rows.append(row)
    return pd.DataFrame(rows)

print("metric helpers ready")

---
## 6. Model and cross-validation driver

The siamese ConvNeXt is preserved unchanged from the working notebook: one shared-weight
backbone runs over the pre image and the post image, the two pooled feature vectors are
concatenated, and a 512-unit head classifies. Sharing the weights is what makes it siamese
— the same filters see both timepoints, so the head compares like with like.

The driver is extended to record, per fold: per-epoch training and validation accuracy and
loss, out-of-fold **probabilities** (needed for ROC-AUC), the best epoch, the number of
epochs actually run, and whether early stopping fired.

In [ ]:
class SiameseClassifier(nn.Module):
    """Shared-weight backbone over (pre, post) -> concat -> MLP head."""
    def __init__(self, model_name, num_classes, use_pair=True, dropout=DROPOUT,
                 img_size=None, pretrained=True):
        super().__init__()
        self.use_pair = use_pair
        kw = dict(pretrained=pretrained, num_classes=0)
        if img_size is not None and any(t in model_name for t in ('vit', 'deit', 'beit')):
            kw['img_size'] = img_size          # CNNs accept any input size as-is
        self.backbone = timm.create_model(model_name, **kw)
        feat = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(feat * (2 if use_pair else 1), 512), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(512, num_classes))

    def forward(self, pre_img, post_img):
        f_post = self.backbone(post_img)        # post first - Grad-CAM relies on this
        if self.use_pair:
            f_pre = self.backbone(pre_img)      # SAME weights -> siamese
            f = torch.cat([f_pre, f_post], dim=1)
        else:
            f = f_post
        return self.classifier(f)


def run_epoch(model, loader, criterion, optimizer=None, scaler=None, desc=""):
    training = optimizer is not None
    model.train() if training else model.eval()
    tot, n, P, Y, PR = 0.0, 0, [], [], []
    with torch.set_grad_enabled(training):
        for pre, post, y in tqdm(loader, desc=desc, leave=False):
            post = post.to(device, non_blocking=True)
            if USE_PAIR:
                pre = pre.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=AMP):
                out = model(pre, post)
                loss = criterion(out, y)
            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
            b = y.size(0); tot += loss.item() * b; n += b
            PR.append(F.softmax(out.float(), 1).detach().cpu().numpy())
            P.append(out.argmax(1).detach().cpu().numpy())
            Y.append(y.detach().cpu().numpy())
    P, Y, PR = np.concatenate(P), np.concatenate(Y), np.concatenate(PR)
    return (tot / n, accuracy_score(Y, P), f1_score(Y, P, average='macro', zero_division=0),
            P, Y, PR)


def run_cv(tag, n_items, labels, n_classes, make_datasets, make_model,
           batch_size, class_names):
    """5-fold CV. Returns per-fold frame, OOF predictions, OOF probabilities,
    per-fold histories and early-stopping metadata."""
    oof_pred = np.full(n_items, -1, dtype=int)
    oof_prob = np.zeros((n_items, n_classes), dtype=np.float32)
    rows, hists = [], {}
    t_all = time.time()

    for fold in range(N_SPLITS):
        print("\n" + "=" * 64)
        print(f"[{tag}] FOLD {fold + 1} / {N_SPLITS}")
        print("=" * 64)
        set_seed(SEED + fold)

        tr_ds, va_ds, va_idx = make_datasets(fold)
        tr_ld = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
        va_ld = DataLoader(va_ds, batch_size=batch_size * 2, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
        print(f"train {len(tr_ds)} (augmented) | validation {len(va_ds)} (no augmentation)")

        model = make_model().to(device)          # FRESH pretrained weights every fold

        w = compute_class_weight('balanced', classes=np.arange(n_classes),
                                 y=labels[np.setdiff1d(np.arange(n_items), va_idx)])
        criterion = nn.CrossEntropyLoss(
            weight=torch.tensor(w, dtype=torch.float32, device=device),
            label_smoothing=LABEL_SMOOTH)
        opt = torch.optim.AdamW([
            {'params': model.backbone.parameters(),   'lr': LR_BACKBONE},
            {'params': model.classifier.parameters(), 'lr': LR_HEAD}],
            weight_decay=WEIGHT_DECAY)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS)
        scaler = torch.amp.GradScaler('cuda', enabled=AMP)

        h = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
        best_f1, best_ep, bad, stopped_early = -1.0, 0, 0, False
        best_pred = best_prob = best_state = None

        for ep in range(1, MAX_EPOCHS + 1):
            t0 = time.time()
            trl, tra, _, _, _, _ = run_epoch(model, tr_ld, criterion, opt, scaler,
                                             desc=f"F{fold+1} e{ep} train")
            vl, va, vf, vp, _, vpr = run_epoch(model, va_ld, criterion,
                                               desc=f"F{fold+1} e{ep} val")
            sched.step()
            h['train_loss'].append(trl); h['train_acc'].append(tra)
            h['val_loss'].append(vl);   h['val_acc'].append(va); h['val_f1'].append(vf)

            mark = ""
            if vf > best_f1:
                best_f1, best_ep = vf, ep
                best_pred, best_prob = vp, vpr
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad, mark = 0, "  <- best"
            else:
                bad += 1
            print(f"  epoch {ep:2d}/{MAX_EPOCHS} | train loss {trl:.4f} acc {tra:.4f} "
                  f"| val loss {vl:.4f} acc {va:.4f} macroF1 {vf:.4f} "
                  f"| {time.time()-t0:.0f}s{mark}")
            if bad >= PATIENCE:
                stopped_early = True
                print(f"  EARLY STOPPING at epoch {ep} "
                      f"(no validation macro-F1 gain for {PATIENCE} epochs)")
                break

        early_stopped = stopped_early
        h['best_epoch'] = best_ep
        h['epochs_run'] = ep
        h['early_stopped'] = early_stopped
        hists[fold] = h

        oof_pred[va_idx] = best_pred
        oof_prob[va_idx] = best_prob
        rows.append({'fold': fold + 1, 'n_train': len(tr_ds), 'n_val': len(va_ds),
                     'best_epoch': best_ep, 'epochs_run': ep,
                     'early_stopped': early_stopped,
                     'best_val_macro_f1': best_f1})
        torch.save(best_state, os.path.join(RESULTS_DIR, f"{tag}_fold{fold+1}.pth"))
        print(f"  --> fold {fold+1}: best epoch {best_ep}, {ep} epochs run, "
              f"early stopping {'YES' if early_stopped else 'NO'}, "
              f"val macro-F1 {best_f1:.4f}")

        del model, opt, tr_ld, va_ld, best_state, tr_ds, va_ds
        gc.collect(); torch.cuda.empty_cache()

    print(f"\n[{tag}] complete in {(time.time()-t_all)/60:.1f} min")
    return pd.DataFrame(rows), oof_pred, oof_prob, hists

print("model and CV driver ready")

---
## 7. Stage 1 — hazard classification

### Fold design, and an honest limitation

Folds are stratified on hazard type at the **tile-pair level**. Grouping folds by disaster
event would be the stronger choice, but it is impossible on this data: earthquake comes
only from `mexico-earthquake` and flood only from `midwest-flooding`, so grouping by event
would place every earthquake pair in one fold and leave four folds with nothing to
validate on.

The consequence, which belongs in the results discussion: two of the four hazard classes
are a single geographic location, so a model can score well by recognising terrain rather
than hazard. Expect a very high Stage 1 score and report it as a property of the benchmark
rather than as a claim about the model. The per-event accuracy breakdown at the end of this
section quantifies it.

In [ ]:
cache_dir = "/kaggle/temp/tile_cache" if os.path.isdir("/kaggle") else "/tmp/tile_cache"
os.makedirs(cache_dir, exist_ok=True)

def cache_one(src, key):
    dst = os.path.join(cache_dir, key + ".jpg")
    if not os.path.exists(dst):
        Image.open(src).convert("RGB").resize((256, 256), Image.BILINEAR).save(dst, quality=95)
    return dst

for df in (dev_df, test_df):
    pre_c, post_c = [], []
    for r in tqdm(df.itertuples(), total=len(df), desc="caching tiles"):
        stem = f"{r.hazard_type}__{r.event_prefix}"
        pre_c.append(cache_one(r.pre_path,  stem + "__pre"))
        post_c.append(cache_one(r.post_path, stem + "__post"))
    df["pre_img"], df["post_img"] = pre_c, post_c
print("cached", 2 * (len(dev_df) + len(test_df)), "tiles")

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
dev_df['fold'] = -1
for f, (_, va) in enumerate(skf.split(dev_df, dev_df['label'])):
    dev_df.loc[va, 'fold'] = f
assert (dev_df['fold'] >= 0).all()

S1_CHECKSUM = checksum(dev_df['event_prefix'], dev_df['fold'])
print("=" * 64)
print("  STAGE 1 FOLD CHECKSUM :", S1_CHECKSUM)
print("  Every architecture must reproduce this exact string.")
print("=" * 64)
print()
print(pd.crosstab(dev_df['fold'] + 1, dev_df['hazard_type'], margins=True))

fold_assign = pd.concat([
    dev_df[['event_prefix', 'hazard_type', 'event_name', 'fold']].assign(split='dev'),
    test_df[['event_prefix', 'hazard_type', 'event_name']].assign(fold=-1, split='test'),
])
fold_assign.to_csv(os.path.join(RESULTS_DIR, "fold_assignments.csv"), index=False)
with open(os.path.join(RESULTS_DIR, "fold_checksums.json"), "w") as fh:
    json.dump({'stage1_checksum': S1_CHECKSUM, 'seed': SEED, 'n_splits': N_SPLITS,
               'n_dev_pairs': len(dev_df), 'n_test_pairs': len(test_df)}, fh, indent=2)
print("\nsaved fold_assignments.csv")

In [ ]:
class TilePairDataset(Dataset):
    def __init__(self, df, tf):
        self.df, self.tf = df.reset_index(drop=True), tf
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        post = self.tf(Image.open(r['post_img']).convert('RGB'))
        pre = self.tf(Image.open(r['pre_img']).convert('RGB')) if USE_PAIR else torch.zeros(1)
        return pre, post, torch.tensor(r['label'], dtype=torch.long)

def s1_make_datasets(fold):
    tr = dev_df[dev_df['fold'] != fold]
    va = dev_df[dev_df['fold'] == fold]
    return (TilePairDataset(tr, s1_train_tf),      # augmented
            TilePairDataset(va, s1_eval_tf),       # deterministic
            dev_df.index[dev_df['fold'] == fold].to_numpy())

def s1_make_model():
    return SiameseClassifier(S1_MODEL, num_classes=4, use_pair=USE_PAIR)

_m = s1_make_model().to(device)
S1_PARAMS_M = round(sum(p.numel() for p in _m.parameters()) / 1e6, 1)
print(S1_MODEL, "| feature dim:", _m.backbone.num_features, "| params (M):", S1_PARAMS_M)
del _m; gc.collect(); torch.cuda.empty_cache()

In [ ]:
s1_res, s1_oof, s1_prob, s1_hist = run_cv(
    tag="stage1_hazard", n_items=len(dev_df), labels=dev_df['label'].values,
    n_classes=4, make_datasets=s1_make_datasets, make_model=s1_make_model,
    batch_size=S1_BATCH, class_names=HAZARD_NAMES)
print()
print(s1_res.to_string(index=False))

### Stage 1 — per-fold results

In [ ]:
y1 = dev_df['label'].values
s1_fold_metrics = []

for f in range(N_SPLITS):
    d = os.path.join(RESULTS_DIR, "stage1_hazard", f"fold_{f+1}")
    idx = dev_df.index[dev_df['fold'] == f].to_numpy()
    yt, yp, pr = y1[idx], s1_oof[idx], s1_prob[idx]

    print("\n" + "#" * 64)
    print(f"# STAGE 1 - FOLD {f+1}")
    print("#" * 64)
    hh = s1_hist[f]
    print(f"best epoch {hh['best_epoch']} | epochs run {hh['epochs_run']}/{MAX_EPOCHS} "
          f"| early stopping {'YES' if hh['early_stopped'] else 'NO'}")

    m = compute_metrics(yt, yp, pr, 4, ordinal=False)
    m['fold'] = f + 1
    m['best_epoch'] = hh['best_epoch']; m['epochs_run'] = hh['epochs_run']
    m['early_stopped'] = hh['early_stopped']
    s1_fold_metrics.append(m)
    print("\n".join(f"  {k:22s} {v:.4f}" for k, v in m.items()
                     if isinstance(v, float)))

    print("\nclassification report")
    print(classification_report(yt, yp, labels=list(range(4)),
                                target_names=HAZARD_NAMES, digits=4, zero_division=0))
    cw = classwise_table(yt, yp, HAZARD_NAMES)
    print(cw.round(4).to_string(index=False))
    cw.to_csv(os.path.join(d, "classwise_metrics.csv"), index=False)

    plot_training_curves(hh, f"Stage 1 fold {f+1}", os.path.join(d, "training_curves.png"))
    cm = plot_confusion(yt, yp, HAZARD_NAMES, f"Stage 1 hazard - fold {f+1}",
                        os.path.join(d, "confusion_matrix.png"))
    pd.DataFrame(cm, index=HAZARD_NAMES, columns=HAZARD_NAMES).to_csv(
        os.path.join(d, "confusion_matrix.csv"))
    aucs = plot_roc(yt, pr, HAZARD_NAMES, f"Stage 1 ROC (one-vs-rest) - fold {f+1}",
                    os.path.join(d, "roc_curves.png"))
    pd.Series(aucs).to_frame('auc').to_csv(os.path.join(d, "roc_auc_values.csv"))

    pd.DataFrame({
        'image_id': dev_df.loc[idx, 'event_prefix'].values,
        'hazard': dev_df.loc[idx, 'hazard_type'].values,
        'ground_truth': [HAZARD_NAMES[i] for i in yt],
        'prediction': [HAZARD_NAMES[i] for i in yp],
        'confidence': pr.max(1), 'fold': f + 1,
        **{f'prob_{n}': pr[:, i] for i, n in enumerate(HAZARD_NAMES)},
    }).to_csv(os.path.join(d, "predictions.csv"), index=False)
    pd.DataFrame([m]).to_csv(os.path.join(d, "metrics.csv"), index=False)
    pd.DataFrame({k: v for k, v in hh.items() if isinstance(v, list)}).to_csv(
        os.path.join(d, "training_history.csv"), index=False)

In [ ]:
S1_KEYS = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'f1_weighted',
           'roc_auc_macro', 'mcc', 'cohen_kappa']
s1_summary = summary_table(s1_fold_metrics, S1_KEYS)
print("STAGE 1 - 5-FOLD SUMMARY")
print(s1_summary.round(4).to_string(index=False))

pd.DataFrame(s1_fold_metrics).to_csv(
    os.path.join(RESULTS_DIR, "stage1_fold_metrics.csv"), index=False)
s1_summary.to_csv(os.path.join(RESULTS_DIR, "stage1_summary.csv"), index=False)

# pooled out-of-fold view over the whole development set
print("\nPooled out-of-fold (all 5 folds combined)")
print(classification_report(y1, s1_oof, labels=list(range(4)),
                            target_names=HAZARD_NAMES, digits=4, zero_division=0))

# the confound diagnostic
d1 = dev_df.copy(); d1['correct'] = (s1_oof == y1).astype(int)
ev = (d1.groupby(['hazard_type', 'event_name'])
        .agg(n=('correct', 'size'), accuracy=('correct', 'mean')).round(4))
print("\nAccuracy by disaster event (a class with one event measures location,")
print("not hazard type - report this as a benchmark limitation):")
print(ev.to_string())
ev.to_csv(os.path.join(RESULTS_DIR, "stage1_hazard", "accuracy_by_event.csv"))

---
## 8. Stage 2 — damage severity

Severity lives in the label JSONs: each building polygon in `*_post_disaster.json` carries
a `properties.subtype`. Each polygon's bounding box, padded by 25% for context, is cut from
both the pre and post tile and classified into one of four severities by the same siamese
ConvNeXt.

### Fold design

Unlike Stage 1, grouped folds are both possible and necessary here. Every damage class
appears in every event, and crops from one tile are near-duplicates of each other, so folds
are grouped on the tile and stratified on severity: **no tile appears in two folds**. This
is asserted below.

### Two documented choices

**`un-classified` is dropped.** It marks a building the annotator could not judge, not a
fifth severity level. Keeping it would ask the model to predict annotator uncertainty.

**The majority class is capped** at `MAX_PER_CLASS` on the development set. The natural
ratio is roughly 80:1 against the rare classes, so uncapped training spends almost all its
time re-learning "intact roof". Selection is deterministic (rank by an MD5 of the crop id,
keep the lowest N) so every architecture selects the identical crops without sharing files.

The cap shifts the class priors, so macro-F1, MCC and QWK are the metrics to report rather
than accuracy. The held-out test set is evaluated **both** capped (comparable with the CV
numbers) and uncapped (the natural distribution).

In [ ]:
def parse_polygons(label_path):
    with open(label_path) as fh:
        data = json.load(fh)
    out = []
    for feat in data.get('features', {}).get('xy', []):
        sub = feat.get('properties', {}).get('subtype', 'un-classified')
        m = re.search(r'\(\((.*?)\)\)', feat.get('wkt', ''))
        if not m:
            continue
        pts = np.array([[float(v) for v in c.strip().split()]
                        for c in m.group(1).split(',')])
        if len(pts) < 3:
            continue
        x0, y0 = pts.min(0); x1, y1 = pts.max(0)
        out.append((sub, x0, y0, x1, y1))
    return out


def build_polys(df):
    rows = []
    for r in tqdm(df.itertuples(), total=len(df), desc="parsing labels"):
        for k, (sub, x0, y0, x1, y1) in enumerate(parse_polygons(r.post_label_path)):
            rows.append({'crop_id': f"{r.event_prefix}#{k}", 'event_prefix': r.event_prefix,
                         'hazard_type': r.hazard_type, 'event_name': r.event_name,
                         'subtype': sub, 'x0': x0, 'y0': y0, 'x1': x1, 'y1': y1,
                         'pre_path': r.pre_path, 'post_path': r.post_path})
    return pd.DataFrame(rows)

dev_poly  = build_polys(dev_df)
test_poly = build_polys(test_df)
print(f"\ndevelopment polygons {len(dev_poly)} | test polygons {len(test_poly)}")
print(dev_poly['subtype'].value_counts().to_string())

In [ ]:
def prepare_crops(poly, cap=None):
    s = poly.copy()
    if DROP_UNCLASSIFIED:
        s = s[s['subtype'].isin(DAMAGE_NAMES)]
    s['label'] = s['subtype'].map(damage_to_idx)
    if cap:
        s['_r'] = s['crop_id'].map(lambda t: hashlib.md5(t.encode()).hexdigest())
        s = (s.sort_values(['label', '_r']).groupby('label', group_keys=False)
               .head(cap).sort_values('_r').drop(columns=['_r']))
    return s.reset_index(drop=True)

sel      = prepare_crops(dev_poly,  cap=MAX_PER_CLASS)   # CV pool
test_sel = prepare_crops(test_poly, cap=None)            # natural distribution
sel['slot'] = sel.index
test_sel['slot'] = test_sel.index

print(f"development crops {len(sel)} (capped at {MAX_PER_CLASS}/class)")
print(sel['subtype'].value_counts().to_string())
print(f"\ntest crops {len(test_sel)} (uncapped, natural distribution)")
print(test_sel['subtype'].value_counts().to_string())

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
sel['fold'] = -1
for f, (_, va) in enumerate(sgkf.split(sel, y=sel['label'], groups=sel['event_prefix'])):
    sel.loc[va, 'fold'] = f
assert (sel['fold'] >= 0).all()

# leakage assertions
for f in range(N_SPLITS):
    tr_g = set(sel.loc[sel.fold != f, 'event_prefix'])
    va_g = set(sel.loc[sel.fold == f, 'event_prefix'])
    assert not (tr_g & va_g), f"group leakage in fold {f+1}"
print("[OK] No group leakage - no tile appears in both train and validation of any fold")
assert not (set(sel['event_prefix']) & set(test_sel['event_prefix'])), \
    "test tiles leaked into the CV pool"
print("[OK] Test set untouched - test crops come only from the held-out tiles")

S2_CHECKSUM = checksum(sel['crop_id'], sel['fold'])
print("\n" + "=" * 64)
print("  STAGE 2 FOLD CHECKSUM :", S2_CHECKSUM)
print("=" * 64)
print()
print(pd.crosstab(sel['fold'] + 1, sel['subtype'], margins=True))

sel.drop(columns=['pre_path', 'post_path']).to_csv(
    os.path.join(RESULTS_DIR, "stage2_fold_assignments.csv"), index=False)
cs = json.load(open(os.path.join(RESULTS_DIR, "fold_checksums.json")))
cs.update({'stage2_checksum': S2_CHECKSUM, 'n_dev_crops': len(sel),
           'n_test_crops': len(test_sel)})
json.dump(cs, open(os.path.join(RESULTS_DIR, "fold_checksums.json"), "w"), indent=2)

In [ ]:
def padded_box(x0, y0, x1, y1, W, H):
    w, h = x1 - x0, y1 - y0
    px, py = w * BBOX_PAD, h * BBOX_PAD
    bx0, by0 = max(0, x0 - px), max(0, y0 - py)
    bx1, by1 = min(W, x1 + px), min(H, y1 + py)
    if bx1 - bx0 < 8:  bx0, bx1 = max(0, bx0 - 4), min(W, bx1 + 4)
    if by1 - by0 < 8:  by0, by1 = max(0, by0 - 4), min(H, by1 + 4)
    return (int(bx0), int(by0), int(max(bx0 + 1, bx1)), int(max(by0 + 1, by1)))


def extract(s):
    ap = np.zeros((len(s), S2_CROP, S2_CROP, 3), np.uint8)
    aq = np.zeros((len(s), S2_CROP, S2_CROP, 3), np.uint8)
    for prefix, grp in tqdm(s.groupby('event_prefix'), total=s['event_prefix'].nunique(),
                            desc="cropping buildings"):
        r0 = grp.iloc[0]
        po = Image.open(r0['post_path']).convert('RGB')
        pr = Image.open(r0['pre_path']).convert('RGB')
        W, H = po.size
        for r in grp.itertuples():
            b = padded_box(r.x0, r.y0, r.x1, r.y1, W, H)
            aq[r.slot] = np.asarray(po.crop(b).resize((S2_CROP, S2_CROP), Image.BILINEAR))
            ap[r.slot] = np.asarray(pr.crop(b).resize((S2_CROP, S2_CROP), Image.BILINEAR))
    return ap, aq

arr_pre, arr_post = extract(sel)
t_pre, t_post     = extract(test_sel)
print("dev crops:", arr_post.shape, "| test crops:", t_post.shape)
print(f"memory: {(arr_post.nbytes*2 + t_post.nbytes*2)/1e9:.2f} GB")

In [ ]:
class CropDataset(Dataset):
    """train=True applies the 8 dihedral augmentations; train=False is deterministic."""
    def __init__(self, idxs, train, pre_arr=None, post_arr=None, labels=None):
        self.idxs = np.asarray(idxs); self.train = train
        self.pre = arr_pre if pre_arr is None else pre_arr
        self.post = arr_post if post_arr is None else post_arr
        self.labels = sel['label'].values if labels is None else labels
    def __len__(self):
        return len(self.idxs)
    def _prep(self, a):
        t = torch.from_numpy(np.ascontiguousarray(a)).permute(2, 0, 1).float().div_(255)
        return (t - _MEAN_T) / _STD_T
    def __getitem__(self, i):
        j = self.idxs[i]
        post, pre = self.post[j], self.pre[j]
        if self.train:                                   # AUGMENTATION - training only
            k = random.randint(0, 3)
            if k:
                post, pre = np.rot90(post, k), np.rot90(pre, k)
            if random.random() < 0.5:
                post, pre = post[:, ::-1], pre[:, ::-1]
            if random.random() < 0.5:
                post, pre = post[::-1], pre[::-1]
        return self._prep(pre), self._prep(post), torch.tensor(self.labels[j], dtype=torch.long)


def s2_make_datasets(fold):
    tr = sel.index[sel['fold'] != fold].to_numpy()
    va = sel.index[sel['fold'] == fold].to_numpy()
    return CropDataset(tr, True), CropDataset(va, False), va

def s2_make_model():
    return SiameseClassifier(S2_MODEL, num_classes=4, use_pair=USE_PAIR, img_size=S2_CROP)

_m = s2_make_model().to(device)
S2_PARAMS_M = round(sum(p.numel() for p in _m.parameters()) / 1e6, 1)
print(S2_MODEL, "| params (M):", S2_PARAMS_M)
del _m; gc.collect(); torch.cuda.empty_cache()

In [ ]:
s2_res, s2_oof, s2_prob, s2_hist = run_cv(
    tag="stage2_damage", n_items=len(sel), labels=sel['label'].values,
    n_classes=4, make_datasets=s2_make_datasets, make_model=s2_make_model,
    batch_size=S2_BATCH, class_names=DAMAGE_NAMES)
print()
print(s2_res.to_string(index=False))

### 8A. Multi-class severity — per-fold results

Four ordinal classes: `0 = no-damage`, `1 = minor-damage`, `2 = major-damage`,
`3 = destroyed`. QWK is reported here because the ordering is meaningful.

In [ ]:
y2 = sel['label'].values
s2_fold_metrics = []

for f in range(N_SPLITS):
    d = os.path.join(RESULTS_DIR, "stage2_severity_multiclass", f"fold_{f+1}")
    idx = sel.index[sel['fold'] == f].to_numpy()
    yt, yp, pr = y2[idx], s2_oof[idx], s2_prob[idx]

    print("\n" + "#" * 64)
    print(f"# STAGE 2 MULTI-CLASS - FOLD {f+1}")
    print("#" * 64)
    hh = s2_hist[f]
    print(f"best epoch {hh['best_epoch']} | epochs run {hh['epochs_run']}/{MAX_EPOCHS} "
          f"| early stopping {'YES' if hh['early_stopped'] else 'NO'}")

    m = compute_metrics(yt, yp, pr, 4, ordinal=True)     # QWK included
    m['fold'] = f + 1
    m['best_epoch'] = hh['best_epoch']; m['epochs_run'] = hh['epochs_run']
    m['early_stopped'] = hh['early_stopped']
    err = yt != yp
    m['adjacent_error_rate'] = float((np.abs(yt - yp)[err] == 1).mean()) if err.any() else 0.0
    s2_fold_metrics.append(m)
    print("\n".join(f"  {k:22s} {v:.4f}" for k, v in m.items() if isinstance(v, float)))

    print("\nclassification report")
    print(classification_report(yt, yp, labels=list(range(4)),
                                target_names=DAMAGE_NAMES, digits=4, zero_division=0))
    cw = classwise_table(yt, yp, DAMAGE_NAMES)
    print(cw.round(4).to_string(index=False))
    cw.to_csv(os.path.join(d, "classwise_metrics.csv"), index=False)

    plot_training_curves(hh, f"Stage 2 fold {f+1}", os.path.join(d, "training_curves.png"))
    cm = plot_confusion(yt, yp, DAMAGE_NAMES, f"Stage 2 severity - fold {f+1}",
                        os.path.join(d, "confusion_matrix.png"))
    pd.DataFrame(cm, index=DAMAGE_NAMES, columns=DAMAGE_NAMES).to_csv(
        os.path.join(d, "confusion_matrix.csv"))
    aucs = plot_roc(yt, pr, DAMAGE_NAMES, f"Stage 2 ROC (one-vs-rest) - fold {f+1}",
                    os.path.join(d, "roc_curves.png"))
    pd.Series(aucs).to_frame('auc').to_csv(os.path.join(d, "roc_auc_values.csv"))

    pd.DataFrame({
        'crop_id': sel.loc[idx, 'crop_id'].values,
        'hazard': sel.loc[idx, 'hazard_type'].values,
        'ground_truth': [DAMAGE_NAMES[i] for i in yt],
        'prediction': [DAMAGE_NAMES[i] for i in yp],
        'confidence': pr.max(1), 'fold': f + 1,
        **{f'prob_{n}': pr[:, i] for i, n in enumerate(DAMAGE_NAMES)},
    }).to_csv(os.path.join(d, "predictions.csv"), index=False)
    pd.DataFrame([m]).to_csv(os.path.join(d, "metrics.csv"), index=False)
    pd.DataFrame({k: v for k, v in hh.items() if isinstance(v, list)}).to_csv(
        os.path.join(d, "training_history.csv"), index=False)

### 8B. Binary severity

**Mapping.** The multi-class predictions are collapsed, not re-trained:

```
0 = no-damage        <- no-damage
1 = damaged          <- minor-damage OR major-damage OR destroyed
```

The probability of `damaged` is `1 - P(no-damage)`, which is what the ROC-AUC is computed
from.

**Why report both views.** The multi-class task asks *how badly* a building is damaged;
the binary task asks *whether* it is damaged at all. For rescue prioritisation the binary
question is the triage decision — send a team or do not — and a model can be excellent at
triage while still confusing minor with major damage. Reporting only the 4-class number
would hide that.

QWK is deliberately **not** reported here. With two classes there is no ordinal distance
for the quadratic weighting to act on, so QWK degenerates to ordinary Cohen's kappa and
reporting it separately would be misleading.

In [ ]:
def to_binary(y, prob):
    yb = (np.asarray(y) > 0).astype(int)
    pd_dmg = 1.0 - prob[:, 0]
    pb = np.stack([prob[:, 0], pd_dmg], 1)
    return yb, (pd_dmg > 0.5).astype(int), pb

s2b_fold_metrics = []
for f in range(N_SPLITS):
    d = os.path.join(RESULTS_DIR, "stage2_severity_binary", f"fold_{f+1}")
    idx = sel.index[sel['fold'] == f].to_numpy()
    yb, pb_pred, pb = to_binary(y2[idx], s2_prob[idx])

    print("\n" + "#" * 64)
    print(f"# STAGE 2 BINARY - FOLD {f+1}")
    print("#" * 64)
    m = compute_metrics(yb, pb_pred, pb, 2, ordinal=False)   # no QWK, by design
    m['fold'] = f + 1
    s2b_fold_metrics.append(m)
    print("\n".join(f"  {k:22s} {v:.4f}" for k, v in m.items() if isinstance(v, float)))

    print("\nclassification report")
    print(classification_report(yb, pb_pred, labels=[0, 1],
                                target_names=BINARY_NAMES, digits=4, zero_division=0))
    cw = classwise_table(yb, pb_pred, BINARY_NAMES)
    print(cw.round(4).to_string(index=False))
    cw.to_csv(os.path.join(d, "classwise_metrics.csv"), index=False)

    cm = plot_confusion(yb, pb_pred, BINARY_NAMES, f"Binary severity - fold {f+1}",
                        os.path.join(d, "confusion_matrix.png"))
    pd.DataFrame(cm, index=BINARY_NAMES, columns=BINARY_NAMES).to_csv(
        os.path.join(d, "confusion_matrix.csv"))
    aucs = plot_roc(yb, pb, BINARY_NAMES, f"Binary severity ROC - fold {f+1}",
                    os.path.join(d, "roc_curve.png"))
    pd.Series(aucs).to_frame('auc').to_csv(os.path.join(d, "roc_auc_values.csv"))

    pd.DataFrame({
        'crop_id': sel.loc[idx, 'crop_id'].values,
        'ground_truth': [BINARY_NAMES[i] for i in yb],
        'prediction': [BINARY_NAMES[i] for i in pb_pred],
        'prob_damaged': pb[:, 1], 'confidence': pb.max(1), 'fold': f + 1,
    }).to_csv(os.path.join(d, "predictions.csv"), index=False)
    pd.DataFrame([m]).to_csv(os.path.join(d, "metrics.csv"), index=False)

In [ ]:
S2_KEYS  = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'f1_weighted',
            'roc_auc_macro', 'qwk', 'mcc', 'cohen_kappa']
S2B_KEYS = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro',
            'roc_auc', 'mcc', 'cohen_kappa']

s2_summary  = summary_table(s2_fold_metrics,  S2_KEYS)
s2b_summary = summary_table(s2b_fold_metrics, S2B_KEYS)

print("STAGE 2 MULTI-CLASS - 5-FOLD SUMMARY")
print(s2_summary.round(4).to_string(index=False))
print("\nSTAGE 2 BINARY - 5-FOLD SUMMARY")
print(s2b_summary.round(4).to_string(index=False))

pd.DataFrame(s2_fold_metrics).to_csv(
    os.path.join(RESULTS_DIR, "stage2_multiclass_fold_metrics.csv"), index=False)
s2_summary.to_csv(os.path.join(RESULTS_DIR, "stage2_multiclass_summary.csv"), index=False)
pd.DataFrame(s2b_fold_metrics).to_csv(
    os.path.join(RESULTS_DIR, "stage2_binary_fold_metrics.csv"), index=False)
s2b_summary.to_csv(os.path.join(RESULTS_DIR, "stage2_binary_summary.csv"), index=False)

q = [m['qwk'] for m in s2_fold_metrics]
print(f"\nQWK per fold: {np.round(q, 4).tolist()}")
print(f"QWK mean {np.mean(q):.4f} +/- {np.std(q, ddof=1):.4f}")
mc = [m['mcc'] for m in s2_fold_metrics]
print(f"MCC mean {np.mean(mc):.4f} +/- {np.std(mc, ddof=1):.4f}")

---
## 9. Explainability — Grad-CAM

**What Grad-CAM contributes to Disaster XAI-Net.** A severity score alone is not actionable
for a rescue coordinator, who has to decide whether to trust it. Grad-CAM answers "which
pixels drove this prediction" by weighting the final convolutional feature maps by the
gradient of the predicted class score with respect to them, then rectifying and upsampling.
A model attending to the collapsed roof is trustworthy; one attending to a road or a
shadow is not, even when both give the same answer.

**EfficientNet compatibility.** EfficientNet is fully convolutional, so Grad-CAM applies
directly with no reshape or architectural surgery — unlike a transformer, whose token
sequences have to be folded back onto a grid first. Two details matter here:

* **Target layer.** Stage 1 hooks `backbone.conv_head`, the final 1x1 convolution before
  global pooling, which at a 224px input gives a 7x7 map. EfficientNet-B0 reduces
  spatially by 32x overall, so on a 64px Stage 2 crop that same layer would yield a 2x2
  map — too coarse to read. Stage 2 therefore hooks `backbone.blocks[2]`, which sits after
  three stride-2 reductions and gives an 8x8 map. This is a resolution trade-off, not a
  correctness issue, and it is stated here so the figures can be interpreted honestly.
* **Siamese forward order.** The backbone is called twice per sample. `forward` calls the
  post image first, so the first captured activation and the last captured gradient both
  belong to the post-disaster branch — which is the one carrying the damage evidence.

Grad-CAM needs gradients, so this section runs outside `no_grad` and outside autocast.

In [ ]:
class GradCAM:
    """Grad-CAM for the siamese model, attributed to the POST-disaster branch."""
    def __init__(self, model, target_layer):
        self.model = model
        self.acts, self.grads = [], []
        self.h1 = target_layer.register_forward_hook(
            lambda m, i, o: self.acts.append(o.detach()))
        self.h2 = target_layer.register_full_backward_hook(
            lambda m, gi, go: self.grads.append(go[0].detach()))

    def remove(self):
        self.h1.remove(); self.h2.remove()

    def __call__(self, pre, post, cls=None):
        self.acts.clear(); self.grads.clear()
        self.model.zero_grad()
        out = self.model(pre, post)                     # post called first
        prob = F.softmax(out.float(), 1)
        if cls is None:
            cls = out.argmax(1)
        out.gather(1, cls.view(-1, 1)).sum().backward()
        A = self.acts[0]                                # forward order: post, then pre
        G = self.grads[-1]                              # backward is reversed -> post last
        w = G.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * A).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=post.shape[-2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze(1)
        flat = cam.flatten(1)
        cam = ((cam - flat.min(1)[0].view(-1, 1, 1)) /
               (flat.max(1)[0].view(-1, 1, 1) - flat.min(1)[0].view(-1, 1, 1) + 1e-8))
        return cam.cpu().numpy(), cls.cpu().numpy(), prob.detach().cpu().numpy()


def denorm(t):
    x = t.cpu() * _STD_T + _MEAN_T
    return x.permute(1, 2, 0).numpy().clip(0, 1)


def gradcam_panel(model, target_layer, samples, names, title, path):
    """samples: list of (pre_tensor, post_tensor, true_label, tag)"""
    cam_engine = GradCAM(model, target_layer)
    rows = len(samples)
    fig, axes = plt.subplots(rows, 4, figsize=(13, 3.1 * rows))
    if rows == 1:
        axes = axes[None, :]
    for r, (pre, post, yt, tag) in enumerate(samples):
        cam, cls, prob = cam_engine(pre.unsqueeze(0).to(device),
                                    post.unsqueeze(0).to(device))
        cam, cls, conf = cam[0], int(cls[0]), float(prob[0].max())
        pre_i, post_i = denorm(pre), denorm(post)
        axes[r, 0].imshow(pre_i);  axes[r, 0].set_title("pre-disaster", fontsize=9)
        axes[r, 1].imshow(post_i); axes[r, 1].set_title("post-disaster", fontsize=9)
        axes[r, 2].imshow(cam, cmap='jet'); axes[r, 2].set_title("Grad-CAM", fontsize=9)
        axes[r, 3].imshow(post_i); axes[r, 3].imshow(cam, cmap='jet', alpha=0.45)
        axes[r, 3].set_title("overlay", fontsize=9)
        ok = "CORRECT" if cls == yt else "WRONG"
        axes[r, 0].set_ylabel(f"{tag}\ntrue: {names[yt]}\npred: {names[cls]} "
                              f"({conf:.2f})\n{ok}", fontsize=8)
        for c in range(4):
            axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
    cam_engine.remove()
    plt.suptitle(title, y=1.005)
    plt.tight_layout(); plt.savefig(path, dpi=150, bbox_inches='tight'); plt.show()

print("Grad-CAM ready")

In [ ]:
GC_DIR = os.path.join(RESULTS_DIR, "gradcam")

# ---- Stage 2 severity Grad-CAM (fold 1 model, fold 1 validation crops) ----
model = s2_make_model().to(device)
model.load_state_dict(torch.load(os.path.join(RESULTS_DIR, "stage2_damage_fold1.pth"),
                                 map_location=device))
model.eval()

idx  = sel.index[sel['fold'] == 0].to_numpy()
yt_, yp_, pr_ = y2[idx], s2_oof[idx], s2_prob[idx]
conf_ = pr_.max(1)
correct = yt_ == yp_

picks = []
if correct.any():
    j = idx[np.where(correct)[0][int(np.argmax(conf_[correct]))]]
    picks.append((j, "high-confidence correct"))
    cj = np.where(correct)[0]
    picks.append((idx[cj[len(cj) // 2]], "typical correct"))
if (~correct).any():
    w = np.where(~correct)[0]
    picks.append((idx[w[int(np.argmax(conf_[~correct]))]], "high-confidence WRONG"))
    picks.append((idx[w[len(w) // 2]], "typical incorrect"))

ds = CropDataset([p[0] for p in picks], train=False)
samples = [(ds[i][0], ds[i][1], int(ds[i][2]), picks[i][1]) for i in range(len(picks))]

gradcam_panel(model, model.backbone.blocks[2], samples, DAMAGE_NAMES,
              "Stage 2 severity - Grad-CAM (EfficientNet block 2 features, 8x8)",
              os.path.join(GC_DIR, "stage2_severity_gradcam.png"))
del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ---- Stage 1 hazard Grad-CAM (fold 1 model, 224px tiles -> 7x7 map) ----
model = s1_make_model().to(device)
model.load_state_dict(torch.load(os.path.join(RESULTS_DIR, "stage1_hazard_fold1.pth"),
                                 map_location=device))
model.eval()

idx = dev_df.index[dev_df['fold'] == 0].to_numpy()
yt_, yp_ = y1[idx], s1_oof[idx]
picks = []
for h in range(4):
    cand = idx[(yt_ == h)]
    if len(cand):
        picks.append((cand[0], f"{HAZARD_NAMES[h]} sample"))

ds = TilePairDataset(dev_df.loc[[p[0] for p in picks]], s1_eval_tf)
samples = [(ds[i][0], ds[i][1], int(ds[i][2]), picks[i][1]) for i in range(len(picks))]

gradcam_panel(model, model.backbone.conv_head, samples, HAZARD_NAMES,
              "Stage 1 hazard - Grad-CAM (EfficientNet conv_head, 7x7)",
              os.path.join(GC_DIR, "stage1_hazard_gradcam.png"))
del model; gc.collect(); torch.cuda.empty_cache()

---
## 10. FINAL HELD-OUT TEST EVALUATION

Everything above is **5-FOLD CROSS-VALIDATION RESULTS**. Everything below is **FINAL
HELD-OUT TEST RESULTS**. They answer different questions and must not be mixed in a table.

**Procedure.** A fresh model is trained from pretrained weights on the *entire* development
set — all 2131 pairs, no fold held back — and evaluated exactly once on the untouched test
set.

**How the epoch count is chosen without touching the test set.** Early stopping needs a
validation split, and carving one out of the test set would destroy it. So the final model
trains for a fixed number of epochs equal to the **mean best epoch across the five CV
folds**, rounded. The epoch count is therefore selected entirely from development data,
which is the standard refit protocol after cross-validation.

**Test-time processing.** Deterministic preprocessing only — resize, centre crop, tensor,
normalise. No augmentation, no test-time augmentation, no threshold tuning.

In [ ]:
S1_FINAL_EPOCHS = int(round(np.mean([h['best_epoch'] for h in s1_hist.values()])))
S2_FINAL_EPOCHS = int(round(np.mean([h['best_epoch'] for h in s2_hist.values()])))
print(f"Stage 1 best epochs per fold: {[h['best_epoch'] for h in s1_hist.values()]}"
      f" -> final model trains {S1_FINAL_EPOCHS} epochs")
print(f"Stage 2 best epochs per fold: {[h['best_epoch'] for h in s2_hist.values()]}"
      f" -> final model trains {S2_FINAL_EPOCHS} epochs")


def train_final(make_model, train_ds, n_classes, labels, epochs, batch_size, tag):
    set_seed(SEED)
    model = make_model().to(device)
    ld = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    w = compute_class_weight('balanced', classes=np.arange(n_classes), y=labels)
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(w, dtype=torch.float32, device=device),
        label_smoothing=LABEL_SMOOTH)
    opt = torch.optim.AdamW([
        {'params': model.backbone.parameters(),   'lr': LR_BACKBONE},
        {'params': model.classifier.parameters(), 'lr': LR_HEAD}], weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    for ep in range(1, epochs + 1):
        trl, tra, _, _, _, _ = run_epoch(model, ld, criterion, opt, scaler,
                                         desc=f"{tag} final e{ep}")
        sched.step()
        print(f"  epoch {ep:2d}/{epochs} | train loss {trl:.4f} acc {tra:.4f}")
    torch.save(model.state_dict(), os.path.join(RESULTS_DIR, f"{tag}_FINAL.pth"))
    return model


@torch.no_grad()
def evaluate(model, ds, batch_size):
    model.eval()
    ld = DataLoader(ds, batch_size=batch_size * 2, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True)
    P, Y = [], []
    for pre, post, y in tqdm(ld, desc="test inference", leave=False):
        post = post.to(device)
        pre = pre.to(device) if USE_PAIR else pre
        with torch.amp.autocast('cuda', enabled=AMP):
            out = model(pre, post)
        P.append(F.softmax(out.float(), 1).cpu().numpy()); Y.append(y.numpy())
    P, Y = np.concatenate(P), np.concatenate(Y)
    return Y, P.argmax(1), P

In [ ]:
# ---------------- Stage 1 final ----------------
print("=" * 64); print("FINAL STAGE 1 MODEL - training on the full development set")
print("=" * 64)
m1 = train_final(s1_make_model, TilePairDataset(dev_df, s1_train_tf), 4,
                 dev_df['label'].values, S1_FINAL_EPOCHS, S1_BATCH, "stage1_hazard")

yt, yp, pr = evaluate(m1, TilePairDataset(test_df, s1_eval_tf), S1_BATCH)
d = os.path.join(RESULTS_DIR, "final_test")
print("\n" + "=" * 64); print("FINAL HELD-OUT TEST RESULTS - STAGE 1 HAZARD")
print("=" * 64)
t1 = compute_metrics(yt, yp, pr, 4, ordinal=False)
print("\n".join(f"  {k:22s} {v:.4f}" for k, v in t1.items()))
print("\n" + classification_report(yt, yp, labels=list(range(4)),
                                    target_names=HAZARD_NAMES, digits=4, zero_division=0))
classwise_table(yt, yp, HAZARD_NAMES).to_csv(
    os.path.join(d, "stage1_classwise.csv"), index=False)
plot_confusion(yt, yp, HAZARD_NAMES, "FINAL TEST - Stage 1 hazard",
               os.path.join(d, "stage1_confusion.png"))
plot_roc(yt, pr, HAZARD_NAMES, "FINAL TEST - Stage 1 ROC (one-vs-rest)",
         os.path.join(d, "stage1_roc.png"))
pd.DataFrame({'image_id': test_df['event_prefix'].values,
              'hazard': test_df['hazard_type'].values,
              'ground_truth': [HAZARD_NAMES[i] for i in yt],
              'prediction': [HAZARD_NAMES[i] for i in yp],
              'confidence': pr.max(1),
              **{f'prob_{n}': pr[:, i] for i, n in enumerate(HAZARD_NAMES)}}
             ).to_csv(os.path.join(d, "stage1_predictions.csv"), index=False)
del m1; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ---------------- Stage 2 final ----------------
print("=" * 64); print("FINAL STAGE 2 MODEL - training on the full development set")
print("=" * 64)
m2 = train_final(s2_make_model, CropDataset(sel.index.to_numpy(), True), 4,
                 sel['label'].values, S2_FINAL_EPOCHS, S2_BATCH, "stage2_damage")

test_ds = CropDataset(np.arange(len(test_sel)), False,
                      pre_arr=t_pre, post_arr=t_post, labels=test_sel['label'].values)
yt, yp, pr = evaluate(m2, test_ds, S2_BATCH)

print("\n" + "=" * 64)
print("FINAL HELD-OUT TEST RESULTS - STAGE 2 MULTI-CLASS (natural distribution)")
print("=" * 64)
t2 = compute_metrics(yt, yp, pr, 4, ordinal=True)
print("\n".join(f"  {k:22s} {v:.4f}" for k, v in t2.items()))
print("\n" + classification_report(yt, yp, labels=list(range(4)),
                                    target_names=DAMAGE_NAMES, digits=4, zero_division=0))
classwise_table(yt, yp, DAMAGE_NAMES).to_csv(
    os.path.join(d, "stage2_multiclass_classwise.csv"), index=False)
plot_confusion(yt, yp, DAMAGE_NAMES, "FINAL TEST - Stage 2 severity",
               os.path.join(d, "stage2_multiclass_confusion.png"))
plot_roc(yt, pr, DAMAGE_NAMES, "FINAL TEST - Stage 2 ROC (one-vs-rest)",
         os.path.join(d, "stage2_multiclass_roc.png"))

# comparable view: same per-class cap as the CV pool
capped = prepare_crops(test_poly, cap=MAX_PER_CLASS)
mask = test_sel['crop_id'].isin(set(capped['crop_id'])).values
t2c = compute_metrics(yt[mask], yp[mask], pr[mask], 4, ordinal=True)
print(f"\nSame test set with the same per-class cap rule applied ({mask.sum()} crops).")
print("Closer to the CV setting than the natural-distribution numbers above, though")
print("the rare classes are too small in the test split for the cap to fully match them:")
print("\n".join(f"  {k:22s} {v:.4f}" for k, v in t2c.items()))

# binary view
yb, pbp, pb = to_binary(yt, pr)
print("\n" + "=" * 64); print("FINAL HELD-OUT TEST RESULTS - STAGE 2 BINARY")
print("=" * 64)
t2b = compute_metrics(yb, pbp, pb, 2, ordinal=False)
print("\n".join(f"  {k:22s} {v:.4f}" for k, v in t2b.items()))
print("\n" + classification_report(yb, pbp, labels=[0, 1],
                                    target_names=BINARY_NAMES, digits=4, zero_division=0))
classwise_table(yb, pbp, BINARY_NAMES).to_csv(
    os.path.join(d, "stage2_binary_classwise.csv"), index=False)
plot_confusion(yb, pbp, BINARY_NAMES, "FINAL TEST - binary severity",
               os.path.join(d, "stage2_binary_confusion.png"))
plot_roc(yb, pb, BINARY_NAMES, "FINAL TEST - binary severity ROC",
         os.path.join(d, "stage2_binary_roc.png"))

pd.DataFrame({'crop_id': test_sel['crop_id'].values,
              'hazard': test_sel['hazard_type'].values,
              'ground_truth': [DAMAGE_NAMES[i] for i in yt],
              'prediction': [DAMAGE_NAMES[i] for i in yp],
              'gt_binary': [BINARY_NAMES[i] for i in yb],
              'pred_binary': [BINARY_NAMES[i] for i in pbp],
              'confidence': pr.max(1), 'prob_damaged': pb[:, 1],
              **{f'prob_{n}': pr[:, i] for i, n in enumerate(DAMAGE_NAMES)}}
             ).to_csv(os.path.join(d, "stage2_predictions.csv"), index=False)

pd.DataFrame([{'stage': 'stage1_hazard', **t1},
              {'stage': 'stage2_multiclass_natural', **t2},
              {'stage': 'stage2_multiclass_capped', **t2c},
              {'stage': 'stage2_binary', **t2b}]).to_csv(
    os.path.join(RESULTS_DIR, "final_test_metrics.csv"), index=False)
del m2; gc.collect(); torch.cuda.empty_cache()

---
## 11. Consolidated summary

In [ ]:
print("=" * 74)
print(f"  DISASTER XAI-NET — {ARCH_TAG.upper()} — CONSOLIDATED RESULTS")
print("=" * 74)
print(f"  stage 1 / stage 2 backbone : {S1_MODEL} / {S2_MODEL}")
print(f"  params (M)                 : {S1_PARAMS_M} / {S2_PARAMS_M}")
print(f"  MAX_EPOCHS / patience      : {MAX_EPOCHS} / {PATIENCE}")
print(f"  development pairs / crops  : {len(dev_df)} / {len(sel)}")
print(f"  held-out test pairs / crops: {len(test_df)} / {len(test_sel)}")
print(f"  STAGE 1 FOLD CHECKSUM      : {S1_CHECKSUM}")
print(f"  STAGE 2 FOLD CHECKSUM      : {S2_CHECKSUM}")
print("-" * 74)
print("  early stopping per fold:")
for f in range(N_SPLITS):
    a, b = s1_hist[f], s2_hist[f]
    print(f"    fold {f+1}: stage1 best ep {a['best_epoch']:2d} of {a['epochs_run']:2d} "
          f"({'ES' if a['early_stopped'] else 'full 40'})   "
          f"stage2 best ep {b['best_epoch']:2d} of {b['epochs_run']:2d} "
          f"({'ES' if b['early_stopped'] else 'full 40'})")
print("=" * 74)

print("\n5-FOLD CROSS-VALIDATION RESULTS")
print("\nStage 1 hazard");                 print(s1_summary.round(4).to_string(index=False))
print("\nStage 2 multi-class severity");   print(s2_summary.round(4).to_string(index=False))
print("\nStage 2 binary severity");        print(s2b_summary.round(4).to_string(index=False))

print("\n\nFINAL HELD-OUT TEST RESULTS")
print(pd.read_csv(os.path.join(RESULTS_DIR, "final_test_metrics.csv")).round(4).to_string(index=False))

pd.DataFrame([{'arch': ARCH_TAG, 's1_model': S1_MODEL, 's2_model': S2_MODEL,
               's1_params_m': S1_PARAMS_M, 's2_params_m': S2_PARAMS_M,
               'lr_backbone': LR_BACKBONE, 'max_epochs': MAX_EPOCHS, 'patience': PATIENCE,
               's1_checksum': S1_CHECKSUM, 's2_checksum': S2_CHECKSUM,
               'n_dev_pairs': len(dev_df), 'n_test_pairs': len(test_df),
               'n_dev_crops': len(sel), 'n_test_crops': len(test_sel),
               's1_folds': json.dumps([m['f1_macro'] for m in s1_fold_metrics]),
               's2_folds': json.dumps([m['f1_macro'] for m in s2_fold_metrics]),
               's2b_folds': json.dumps([m['f1_macro'] for m in s2b_fold_metrics]),
               's2_qwk_folds': json.dumps([m['qwk'] for m in s2_fold_metrics])}]
             ).to_csv(os.path.join(RESULTS_DIR, f"TEAM_REPORT_{ARCH_TAG}.csv"), index=False)

print("\n\nfiles written:")
for root, _, files in os.walk(RESULTS_DIR):
    if files:
        print(" ", root.replace(RESULTS_DIR, "effnet_results"))
        for fn in sorted(files)[:8]:
            print("     ", fn)
        if len(files) > 8:
            print(f"      ... {len(files)-8} more")

---
## 12. Notes for the viva

**Why 80/20 first.** Cross-validation consumes its validation folds through early stopping
and best-epoch selection, so CV scores are mildly optimistic. A test set that influenced
none of those decisions gives an unbiased final estimate.

**Why 5-fold on the 80% only.** Folding the test set in would let it shape early stopping
and model selection, which is exactly what holding it out was for.

**Why pre/post pairs stay together.** A manifest row *is* a pair, so both images share a
fold by construction. Splitting them would let the model see a tile's before image in
training and its after image in validation — the same scene, so validation would be
measuring memorisation.

**Why training is augmented and validation is not.** Augmentation enlarges the training
distribution. At validation time it makes the score depend on random draws, so early
stopping would fire on noise and the metric would describe distorted images rather than
real ones.

**Why max 40 epochs with early stopping.** 40 is an upper bound generous enough that the
model is never cut off mid-improvement; early stopping ends a fold once validation
macro-F1 has stalled for 8 epochs, which prevents overfitting and saves compute. Each
fold's best epoch, epochs run, and early-stopping flag are recorded.

**Why QWK for severity but not for hazard.** Severity is ordinal — no-damage < minor <
major < destroyed — so confusing adjacent classes is a smaller error than confusing distant
ones, and the quadratic weighting encodes that. Hazard types have no ordering, so any
distance between cyclone and flood would be invented.

**What MCC and kappa measure.** MCC is a correlation between true and predicted labels,
-1 to +1, which cannot be inflated by predicting the majority class. Cohen's kappa is
agreement corrected for chance agreement at the observed class rates. Both are reported
because accuracy is misleading under imbalance.

**Binary versus multi-class severity.** Multi-class asks how badly a building is damaged;
binary (`damaged = minor OR major OR destroyed`) asks whether it is damaged at all, which
is the triage decision for rescue prioritisation. The binary view is derived by collapsing
the same model's predictions, not by training a second model, so the two views describe one
system.

**How the folds are compared.** Every fold reports the full metric set, and the summary
tables give metric x fold x mean x std. Because every architecture reads the same fold
assignment (verified by checksum), models are compared **per fold** rather than by their
averages — with a fold-to-fold spread of a few points, a model is only better if it leads
on at least four of five folds.

**How the final test result is obtained.** A fresh model is trained on the entire
development set for the mean best epoch from CV — an epoch count chosen without ever
reading the test set — then evaluated once.

**What Grad-CAM adds.** It shows which pixels drove each severity prediction, letting a
coordinator distinguish a model attending to a collapsed roof from one attending to a
shadow. Correct, incorrect and high-confidence-wrong examples are all shown, because the
failure cases are the informative ones.

### A limitation to state, not hide

Stage 1 will score near-perfectly. Earthquake comes only from `mexico-earthquake` and flood
only from `midwest-flooding`, so two of four hazard classes are a single geographic
location and the model can separate terrain rather than hazard. The per-event accuracy
table in section 7 is the evidence. Report Stage 1 as a saturated, confounded task and let
Stage 2 carry the model comparison — noting a benchmark limitation you found yourself is
worth more than an unexplained 0.99.